# Object Tracking in Videos using YOLOv5 and DeepSORT

This project implements a multi-object tracking pipeline for traffic video footage. YOLOv5 is used for object detection, and DeepSORT is used to assign persistent tracking IDs to objects across frames.

## 1. Install Dependencies

This section installs YOLOv5 and the DeepSORT tracking library required for detection and tracking.

In [ ]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt
!pip install deep-sort-realtime opencv-python-headless

## 2. Upload Input Video

A short traffic video is uploaded as the input. The video contains multiple moving vehicles and motorbikes, which makes it suitable for multi-object tracking.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 3. Load YOLOv5 and DeepSORT

YOLOv5 is loaded as the object detector. DeepSORT is initialized as the tracker to maintain object identities across consecutive video frames.

In [ ]:
import cv2
import torch
import numpy as np
from deep_sort_realtime.deepsort_tracker import DeepSort

# Load YOLOv5 model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# Initialize DeepSORT tracker
tracker = DeepSort(max_age=30)

# Video path
video_path = "12207144_1920_1080_30fps.mp4"

# Open video
cap = cv2.VideoCapture(video_path)

# Video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Output video writer
out = cv2.VideoWriter(
    'tracked_output.mp4',
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

frame_count = 0

## 4. Run Object Detection and Tracking

Each frame is processed using YOLOv5. The detected bounding boxes are passed into DeepSORT, which assigns a unique tracking ID to each object. The final output is saved as a tracked video.

In [ ]:


while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    # YOLO detection
    results = model(frame)

    detections = []

    for *box, conf, cls in results.xyxy[0]:

        x1, y1, x2, y2 = map(int, box)

        confidence = float(conf)
        class_id = int(cls)

        if confidence > 0.4:

            w = x2 - x1
            h = y2 - y1

            detections.append(
                ([x1, y1, w, h], confidence, class_id)
            )

    # DeepSORT tracking
    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:

        if not track.is_confirmed():
            continue

        track_id = track.track_id

        ltrb = track.to_ltrb()

        x1, y1, x2, y2 = map(int, ltrb)

        # Draw box
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # Draw ID
        cv2.putText(
            frame,
            f'ID: {track_id}',
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    out.write(frame)

    frame_count += 1

    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames")

cap.release()
out.release()

print("Tracking complete.")

## 5. Convert Output Video for Browser Playback

The initial output video is converted to H.264 format so it can be played correctly inside the notebook and on GitHub-compatible viewers.

In [ ]:
!ffmpeg -y -i tracked_output.mp4 -vcodec libx264 tracked_output_h264.mp4

## 6. Display Final Tracked Video

The final H.264 video is displayed below. Green bounding boxes show tracked objects, and each ID represents a unique object tracked across frames.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open('tracked_output_h264.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=800 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")

## 7. Download Final Output

The final tracked video is downloaded locally for further analysis and inclusion in the GitHub repository.

In [ ]:
from google.colab import files
files.download('tracked_output_h264.mp4')

## Conclusion

The project successfully detected and tracked multiple vehicles and motorbikes in a traffic video. YOLOv5 handled object detection, while DeepSORT maintained object identities across frames using unique tracking IDs.

The system performed well in a busy traffic scene. Some ID switching may occur when objects overlap, are partially hidden, or leave and re-enter the frame.